<a href="https://colab.research.google.com/github/tsursnake/52025-COURSE-YOHAN-GROSS/blob/main/assignment2/PS1_Part2_Hello_NN_2026_ID_329758296.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PS1: Your first library-free neural network!  

Advanced Learning 2026


For SUBMISSION:   

Please upload the complete and executed `ipynb` to your git repository. Verify that all of your output can be viewed directly from github, and provide a link to that git file below.

~~~
STUDENT ID: 329758296
~~~

~~~
STUDENT GIT LINK: https://github.com/tsursnake/52025-COURSE-YOHAN-GROSS/blob/main/assignment_2/PS1_Part2_Hello_NN_2026_ID_329758296.ipynb
~~~
In Addition, don't forget to add your ID to the files:    
  
`PS1_Part2_Hello_NN_2026_ID_[000000000].html`   


In [1]:
import numpy as np # You are allowed to use  only numpy.
import time


**Welcome**.   

In this part of the problem set you are set to build a complete and flexible neural network.  
This neural network will be library free (in the sense that we won't use PyTorch/Tensorflow/etc.).   

Let's do a quick review of the basic neural-network components:  


*   *Layer* - can be fully connected (dense/hidden), convolution, etc.
  * Forward propagation- the layer outputs the next layer's input
  * Backward propagation- the layer also outputs the gradient descent update
*   *Activation* Layer (e.g. ReLU) - there are no parameters, only gradients with respect to the input. We want to compute both the gradient w.r.t the parameters of the layer and to create the gradient with respect to the layer's inputs
   * *Forward propagation*- the layer outputs the next layer's input
   * *Backward propagation*- the layer also outputs the gradient descent update
*   *Loss Function* : how our model  quantifies the difference between the predicted outputs the actual (target) values  
*   *Network Wrapper*-  wraps our components together as a trainable model.






Useful resource:  
* Gradient descent for neural networks [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/resource/view.php?id=402297).
* Neural network architecture [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/url/view.php?id=402298).

### 0. Loading data

You are going to test and evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.

In [2]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


#### Data transformations





In [3]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255

# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)

# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)


### 1. Network's Components

Please fill-in the missing code in the code boxes below (only where  `#### SOLUTION REQUIRED ####` is specified).   

In [4]:
# This class is a general layer primitive, defining that each instance must
# have an (input,output) parameters, and 2 functions: forward+backward propogation
class Layer_Primitive:
    def __init__(self):
        self.input = None
        self.output = None

    # computes the output Y of a layer for a given input X
    def forward_propagation(self, input):
        raise NotImplementedError

    # computes dE/dX for a given dE/dY (and update parameters if any)
    def backward_propagation(self, output_error, learning_rate):
        raise NotImplementedError


#### Fully Connected Layer

A fully-connected layer (a.k.a. affine, dense,linear layer) connects every input neuron to every output neuron.   
It has 2 parameters: (input, output).   
You need to define (code) the following:
* its initialization weights with random weights.
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

Parameters must be intitialized with some values. There are many ways to initialize the weights, and you are encouraged to do a quick research about the common methods. Any commonly used method will be accepted.  

1.1 (20 pts)

In [5]:
#### SOLUTION REQUIRED ####

# inherit from base class Layer
class Affine_Layer(Layer_Primitive):
    # input_size = number of input neurons
    # output_size = number of output neurons
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2 / (input_size + output_size))  # xavier
        self.bias = np.random.randn(1, output_size) * 0.01


    # returns output for a given input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = input_data @ self.weights + self.bias
        return self.output

    # computes dE/dW, dE/dB for a given output_error=dE/dY. Returns input_error=dE/dX.
    def backward_propagation(self, output_grad, learning_rate):
        input_error = output_grad @ self.weights.T
        weights_error = self.input.T @ output_grad

        # update parameters
        self.weights -= learning_rate * weights_error
        self.bias -= learning_rate * output_grad

        return input_error


#### Activation layers

Activation functions are often a non-linear functions that aid in how well the network model adapts to and learns  the training dataset. The choice of activation function in the output layer will define the type of predictions the model can make.  



In [6]:
# inherit from base class Layer
class ActivationLayer(Layer_Primitive):
    def __init__(self, activation, activation_grad):
        self.activation = activation
        self.activation_grad = activation_grad

    # returns the activated input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = self.activation(self.input)
        return self.output

    # Returns input_error=dE/dX for a given output_grad=dE/dY.
    # learning_rate is not used because there is no "learnable" parameters.
    def backward_propagation(self, output_grad, learning_rate):
        return self.activation_grad(self.input) * output_grad



You need to define (code) the following via different functions:
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

1.2 (20 pts)

In [7]:
#### SOLUTION REQUIRED ####

# activation functions and their derivatives:

def tanh(x):
    return np.tanh(x)

def tanh_grad(x):
    return 1 - np.tanh(x) ** 2

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def sigmoid(x):
    s = 1 / (1 + np.exp(-x))
    return s

def sigmoid_grad(x):
    s = 1 / (1 + np.exp(-x))
    return s * (1 - s)


#### Loss function

1.3 (10 pts)

In [8]:
#### SOLUTION REQUIRED ####

# loss function and its derivative

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_grad(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size


#### Putting everything together

1.4 (10 pts)

In [9]:
#### SOLUTION REQUIRED (in `predict`) ####

class MyNetwork:
    def __init__(self):
        self.layers = []
        self.loss = None
        self.loss_grad = None

    # add layer to network
    def add(self, layer):
        self.layers.append(layer)

    # set loss to use
    def use_loss(self, loss, loss_grad):
        self.loss = loss
        self.loss_grad = loss_grad


    # train the network
    def fit(self, x_train, y_train, epochs, learning_rate):
        # sample dimension first
        samples = len(x_train)

        # training loop
        for i in range(epochs):
            err = 0
            for j in range(samples):
                # forward propagation
                output = x_train[j]
                for layer in self.layers:
                    output = layer.forward_propagation(output)

                # compute loss (for display purpose only)
                err += self.loss(y_train[j], output)

                # backward propagation
                grad = self.loss_grad(y_train[j], output)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, learning_rate)

            # calculate average error on all samples
            err /= samples
            print('Training epoch %d/%d   error=%f' % (i+1, epochs, err))


    # predict output for given input
    def predict(self, x_test,y_test=np.array([])):
        if y_test.size:
           assert len(x_test)==len(y_test) # if Y is given
        # sample dimension first
        samples = len(x_test)
        result = []
        loss = 0
        correct = 0
        # run network over all samples
        for i in range(samples):
            # forward propagation
            output = x_test[i]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            result.append(output)
            # ONLY IF LABELS ARE GIVEN (Y):
            if y_test.size:
                # Evaluate the output against Y,
                # calculate loss against Y, add to `loss`:
                loss += self.loss(y_test[i], output)
                target = y_test[i]
                # Evaluate the label of the output against real, and if identical,
                # add +1 to `correct`:
                if np.argmax(output) == np.argmax(target):
                   correct += 1
        if y_test.size:
            mean_loss = loss/samples

            print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.
                  format(mean_loss, correct, samples,100. * correct / samples))

        return result


## 2. Testing Your Neural Network

### Defining our main neural network architecture

Define your network's architecture:  
(Please rationalize your choice of activation funciton.)
* first affine layer that takes your input and outputs 128 nodes
* `tanh/relu/sigmoid` activation layer following the first affine layer
* second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh/relu/sigmoid` activation layer following the second affine layer
* third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh/relu/sigmoid` activation layer following the last affine layer


2.1 (5 pts)

In [10]:
#### SOLUTION REQUIRED (in `predict`) ####

# Network Architecture
net = MyNetwork()
net.add(Affine_Layer(28 * 28, 128))
net.add(ActivationLayer(tanh, tanh_grad))
net.add(Affine_Layer(128, 64))
net.add(ActivationLayer(tanh, tanh_grad))
net.add(Affine_Layer(64, 10))
net.add(ActivationLayer(sigmoid, sigmoid_grad))


I used tanh in the two hidden layers and sigmoid on the output. tanh is zero-centered and smooth, so the hidden units do not all push the gradients in the same direction like a sigmoid would. The output has 10 nodes and is compared to a one-hot vector with the MSE, so a sigmoid fits well since it keeps every output in (0, 1).

### Training!

In [11]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# as we didn't implemented mini-batch GD, training will be pretty slow if we update at each iteration on 60000 samples...
net.use_loss(mse, mse_grad)
epoch_num = 5 # use your own number of epochs
lr = 0.1       # use your own learning rate, I advise to play around with it and see how results change—be sure to rereun the network above across trial runs.
t1 = time.time()
net.fit(x_train, y_train, epochs=epoch_num, learning_rate=lr) # subset the data if you wish, just make sure it is a working model...
print(f"Total process time: {round(time.time() - t1,3)}")


Training epoch 1/5   error=0.014240
Training epoch 2/5   error=0.007069
Training epoch 3/5   error=0.005232
Training epoch 4/5   error=0.004172
Training epoch 5/5   error=0.003441
Total process time: 192.922


### Evaluation

Exciting! Now is the time to test your model.   

    May the gradients be always in your favor.

In [12]:
output = net.predict(x_test, y_test)


Test set: Avg. loss: 0.0046, Accuracy: 9727/10000 (97%)



## 3. Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

#### Prepare the data as tensors using PyTorch DataLoader:

In [14]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchNet` class with an identical architecture you used in your home-made network.

3.1 (10 pts)

In [15]:
#### SOLUTION REQUIRED  ####

class PyTorchNet(nn.Module):
    def __init__(self):
        super(PyTorchNet, self).__init__()
        input_size = 28 * 28
        num_classes = 10

        self.fc1 = nn.Linear(input_size, 128)
        self.activ1 = nn.Tanh()
        self.fc2 = nn.Linear(128, 64)
        self.activ2 = nn.Tanh()
        self.fc3 = nn.Linear(64, num_classes)
        self.activ3 = nn.Sigmoid()


    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.activ1(self.fc1(x))
        x = self.activ2(self.fc2(x))
        x = self.activ3(self.fc3(x))

        return x

In [16]:
# Train the model
num_epochs = 20
pt_learning_rate = 0.01
pt_network = PyTorchNet()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

pt_t0 = time.time()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
print(f"Total process time: {round(time.time() - pt_t0,3)}")

Epoch [1/20], Step [500/938], Loss: 0.0211
Epoch [2/20], Step [500/938], Loss: 0.0094
Epoch [3/20], Step [500/938], Loss: 0.0057
Epoch [4/20], Step [500/938], Loss: 0.0231
Epoch [5/20], Step [500/938], Loss: 0.0254
Epoch [6/20], Step [500/938], Loss: 0.0146
Epoch [7/20], Step [500/938], Loss: 0.0097
Epoch [8/20], Step [500/938], Loss: 0.0181
Epoch [9/20], Step [500/938], Loss: 0.0084
Epoch [10/20], Step [500/938], Loss: 0.0132
Epoch [11/20], Step [500/938], Loss: 0.0087
Epoch [12/20], Step [500/938], Loss: 0.0082
Epoch [13/20], Step [500/938], Loss: 0.0178
Epoch [14/20], Step [500/938], Loss: 0.0159
Epoch [15/20], Step [500/938], Loss: 0.0056
Epoch [16/20], Step [500/938], Loss: 0.0115
Epoch [17/20], Step [500/938], Loss: 0.0097
Epoch [18/20], Step [500/938], Loss: 0.0101
Epoch [19/20], Step [500/938], Loss: 0.0068
Epoch [20/20], Step [500/938], Loss: 0.0145
Total process time: 63.193


Evaluation:

In [17]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target) * data.size(0)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0110, Accuracy: 9384/10000 (94%)



3.2 (10 pts)

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

Write your answers here:

1. My network did better, even though the architecture is the same. The difference is the training and not the library. My network is updated after every single image (60000 updates per epoch, lr 0.1), while the PyTorch loop uses batches of 64 (938 updates per epoch) with Adam and lr 0.01. That learning rate is too big for Adam here: the printed loss jumps around from step to step instead of going down smoothly, so the final PyTorch model ends up worse.   
2. PyTorch is much faster, per epoch and in total (even though it runs 20 epochs and my network only 5). It does the forward and backward pass on 64 images at once with optimized matrix products, while my code is a python loop that goes through the layers for one image at a time, 60000 tiny matrix products per epoch.   
3. Softmax with cross-entropy on the output instead of sigmoid with MSE (the outputs become probabilities and the gradient does not vanish when the sigmoid saturates), relu in the hidden layers, wider layers, a better initialization (He), and some regularization like dropout. For images, a convolutional network would work better than flattening the pixels.   
4. Use mini-batch gradient descent (faster because it is vectorized, and less noisy than one image at a time), shuffle the data in every epoch (my fit loop always goes through the images in the same order), decay the learning rate, add momentum or use Adam, and use a validation set to pick the number of epochs (early stopping). For the PyTorch model, a smaller learning rate for Adam (around 1e-3).


## 4. The Network Wars!

Here is your chance to play with your model's architecture in order to break your own benchmark set eariler.  
You can add/remove layers, play with their sizes, types, etc.   
You can add a new loss if you wish, or anything else that will fairly give your model an advantage over base.  

4.1 (15 pts)

In [18]:
#### SOLUTION REQUIRED  ####

# same pieces as before, but relu + softmax/cross-entropy and mini-batches
class Batch_Affine(Affine_Layer):
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)  # He init
        self.bias = np.zeros((1, output_size))

    def backward_propagation(self, output_grad, learning_rate):
        input_error = output_grad @ self.weights.T
        self.weights -= learning_rate * (self.input.T @ output_grad)
        self.bias -= learning_rate * output_grad.sum(axis=0, keepdims=True)
        return input_error

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(y_true, logits):
    return -np.mean(np.sum(y_true * np.log(softmax(logits) + 1e-12), axis=-1))

def cross_entropy_grad(y_true, logits):
    return softmax(logits) - y_true

class BatchNetwork(MyNetwork):
    def fit(self, x_train, y_train, epochs, learning_rate, batch_size=128, decay=1.0):
        x = x_train.reshape(len(x_train), -1)
        for i in range(epochs):
            idx = np.random.permutation(len(x))
            err = 0
            for s in range(0, len(x), batch_size):
                b = idx[s:s + batch_size]
                output = x[b]
                for layer in self.layers:
                    output = layer.forward_propagation(output)
                err += self.loss(y_train[b], output) * len(b)
                grad = self.loss_grad(y_train[b], output) / len(b)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, learning_rate)
            learning_rate *= decay
            print('Training epoch %d/%d   error=%f' % (i+1, epochs, err / len(x)))

np.random.seed(1)
net2 = BatchNetwork()
net2.add(Batch_Affine(28 * 28, 256))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Batch_Affine(256, 128))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Batch_Affine(128, 10))
net2.use_loss(cross_entropy, cross_entropy_grad)

t1 = time.time()
net2.fit(x_train, y_train, epochs=15, learning_rate=0.2, batch_size=128, decay=0.85)
print(f"Total process time: {round(time.time() - t1,3)}")
output = net2.predict(x_test, y_test)


Training epoch 1/15   error=0.324364
Training epoch 2/15   error=0.134115
Training epoch 3/15   error=0.095674
Training epoch 4/15   error=0.075366
Training epoch 5/15   error=0.062566
Training epoch 6/15   error=0.052507
Training epoch 7/15   error=0.045898
Training epoch 8/15   error=0.040827
Training epoch 9/15   error=0.036901
Training epoch 10/15   error=0.033945
Training epoch 11/15   error=0.031406
Training epoch 12/15   error=0.029600
Training epoch 13/15   error=0.027980
Training epoch 14/15   error=0.026794
Training epoch 15/15   error=0.025809
Total process time: 41.053

Test set: Avg. loss: 0.0672, Accuracy: 9789/10000 (98%)



What I changed compared to the base network: relu hidden layers (256 and 128 nodes) with He initialization, softmax with cross-entropy instead of sigmoid with MSE, mini-batches of 128 with a shuffle in every epoch, and a learning rate that decays by 0.85 after each epoch. It reaches about 98% test accuracy, better than both the base network and the PyTorch model above, and it trains in under a minute. The loss printed at the end is the cross-entropy, so it can not be compared directly to the MSE of the other models.